# p2 · 정규화 (term → normed_term)

p1의 term(원문 표기)을 합쳐 **normed_term = 식별 키(PK)** 를 만든다. 합친 이유는 전부 로그로 남긴다.

**Gate 2**
1. 결정론 규칙 먼저: 표기(대소문자·구두점·하이픈), 머리명사 단복수, 약어 사전
2. 임베딩은 병합 **후보**만 만든다. 병합은 LLM이 정해진 액션(`MERGE` / `DISTINCT` / `UNSURE`)으로만 결정하고, 액션은 검증 후 실행한다
3. 같은 PK 충돌(이미 `DISTINCT`로 판정된 쌍이 전이적으로 합쳐지는 경우)이나 모호한 병합(`UNSURE`, 낮은 confidence)은 사람 큐로 보낸다
4. 병합 로그 보존

참고: TRACE-KG(임베딩 후보 + 제한된 편집 액션 + 편집 로그), LOM(표면형 규칙 → 도메인 정규화 → 임베딩 순서), GrOIL(표면 변형과 근거 주석), Axiom ontology-master-schema/21(결정론 Identity).

입력: `outputs/p1/terms.json` → 출력: `outputs/p2/`

## 설정

In [1]:
import re
import unicodedata
from collections import Counter, defaultdict
from typing import Literal

import numpy as np
import spacy
from pydantic import BaseModel, Field

from common import EMBEDDING_MODEL, GEMINI_MODEL, OUTPUT_ROOT, gate_record, embed, generate_json, load_chunks, read_json, stage_dir, write_json

P1 = read_json(OUTPUT_ROOT / "p1" / "terms.json")
OUTPUT_DIR = stage_dir("p2")

SIM_THRESHOLD = 0.92   # 이 이상이면 병합 후보 (임베딩은 후보 생성만)
MAX_PAIRS = 150        # LLM 판정에 보낼 후보 쌍 상한
MERGE_CONF = 0.8       # MERGE라도 confidence가 이보다 낮으면 사람 큐

# 약어 사전: 문서에서 "Long Form (ABBR)" 패턴으로 자동 수집 + 수동 보강
MANUAL_ABBREVIATIONS: dict[str, str] = {}

doc_id, doc_text, chunks = load_chunks()
terms = P1["terms"]
print(GEMINI_MODEL, EMBEDDING_MODEL, "p1 terms:", len(terms))

gemini-3.5-flash gemini-embedding-001 p1 terms: 162


## ① 결정론 규칙

`normalize_key`는 순수 함수다. 같은 key가 나온 term은 규칙만으로 합친다.
- NFKC, 소문자, 하이픈·슬래시 → 공백, 그 밖의 구두점 제거
- 약어 → 전개형 (사전에 있을 때만)
- 머리명사(마지막 단어)만 단수화. spaCy가 구 전체를 태깅해 머리명사가 복수 명사(NNS)일 때만 표제어로 바꾼다. `goods`처럼 복수형이 곧 기본형인 단어는 예외

In [2]:
ABBR_RE = re.compile(r"((?:[A-Za-z][\w-]*\s+){1,6}?)\(([A-Z]{2,6})\)")


def collect_abbreviations(text: str) -> dict[str, str]:
    found = {}
    for m in ABBR_RE.finditer(text):
        words, abbr = m.group(1).split(), m.group(2)
        # 약어 글자 수만큼 뒤에서 단어를 잘라 첫 글자가 맞으면 채택
        cand = words[-len(abbr):]
        if len(cand) == len(abbr) and "".join(w[0] for w in cand).upper() == abbr:
            found[abbr.lower()] = " ".join(cand).lower()
    return found


ABBREVIATIONS = {**collect_abbreviations(doc_text), **{k.lower(): v.lower() for k, v in MANUAL_ABBREVIATIONS.items()}}
print("abbreviations:", ABBREVIATIONS)

PLURAL_INVARIANT = {"goods", "services", "series", "status", "analysis", "business", "process", "news", "means", "data"}


nlp = spacy.load("en_core_web_sm", exclude=["parser", "ner"])


def singularize_head(words: list[str]) -> str:
    """구의 마지막 단어를 단수형으로. 구 전체를 태깅해 머리명사가 복수 명사일 때만 표제어를 쓴다."""
    head = words[-1]
    if head in PLURAL_INVARIANT:
        return head
    doc = nlp(" ".join(words))
    last = doc[-1]
    if last.text == head and last.tag_ in ("NNS", "NNPS") and last.lemma_:
        return last.lemma_.lower()
    return head


def normalize_key(span: str) -> tuple[str, list[str]]:
    """(key, 적용된 규칙들)"""
    rules = []
    s = unicodedata.normalize("NFKC", span)
    if s.lower() != s:
        rules.append("casefold")
    s = s.lower()
    t = re.sub(r"[-/]", " ", s)
    t = re.sub(r"[^\w\s]", "", t)
    if t != s:
        rules.append("punctuation")
    tokens = t.split()
    expanded = []
    for tok in tokens:
        if tok in ABBREVIATIONS:
            expanded.extend(ABBREVIATIONS[tok].split())
            rules.append(f"abbreviation:{tok}")
        else:
            expanded.append(tok)
    if expanded:
        head = singularize_head(expanded)
        if head != expanded[-1]:
            rules.append("singular")
        expanded[-1] = head
    return "_".join(expanded), rules


for s in ["Goods receipt", "purchase order items", "2-way match", "EC purchase orders", "invoices", "activities", "recurring invoices", "goods", "process analysis"]:
    print(s, "->", normalize_key(s))

abbreviations: {}


Goods receipt -> ('goods_receipt', ['casefold'])
purchase order items -> ('purchase_order_item', ['singular'])
2-way match -> ('2_way_match', ['punctuation'])
EC purchase orders -> ('ec_purchase_order', ['casefold', 'singular'])
invoices -> ('invoice', ['singular'])
activities -> ('activity', ['singular'])
recurring invoices -> ('recurring_invoice', ['singular'])
goods -> ('goods', [])
process analysis -> ('process_analysis', [])


In [3]:
groups = defaultdict(list)   # key -> [term]
merge_log = []

for t in terms:
    key, rules = normalize_key(t["span"])
    groups[key].append(t)
    t["_key"], t["_rules"] = key, rules

for key, members in groups.items():
    if len(members) > 1:
        merge_log.append({
            "action": "RULE_MERGE",
            "into": key,
            "spans": [m["span"] for m in members],
            "rules": sorted({r for m in members for r in m["_rules"]}),
        })

print("terms:", len(terms), "-> rule groups:", len(groups), "| rule merges:", len(merge_log))
for e in merge_log[:15]:
    print(f"  {e['into']:<35} <- {e['spans']}  {e['rules']}")

terms: 162 -> rule groups: 145 | rule merges: 17
  framework_order                     <- ['framework order', 'framework orders']  ['singular']
  goods_receipt                       <- ['Goods receipt', 'goods receipt']  ['casefold']
  invoice                             <- ['invoice', 'invoices']  ['singular']
  invoice_clearance                   <- ['invoice clearance', 'Invoice clearance']  ['casefold']
  invoice_receipt                     <- ['invoice receipt', 'Invoice receipt']  ['casefold']
  item_case                           <- ['item case', 'item cases']  ['singular']
  item_deletion                       <- ['Item deletion', 'item deletion']  ['casefold']
  order_confirmation                  <- ['order confirmation', 'order confirmations']  ['singular']
  order_item                          <- ['order item', 'order items']  ['singular']
  payment_block_creation              <- ['payment block creation', 'Payment block creation']  ['casefold']
  process_variant           

## ② 임베딩으로 병합 후보 만들기

규칙 그룹마다 대표 표기를 임베딩하고, 코사인 유사도가 `SIM_THRESHOLD` 이상인 쌍만 후보로 올린다. 유사도는 동일성의 증거가 아니다(LOM이 스스로 적은 한계). 결정은 다음 단계에서 한다.

In [4]:
def group_label(members: list[dict]) -> str:
    # 언급이 가장 많은 원문 표기, 동률이면 짧은 것
    counts = Counter()
    for m in members:
        counts[m["span"]] += sum(len(x["offsets"]) or 1 for x in m["mentions"])
    return sorted(counts.items(), key=lambda kv: (-kv[1], len(kv[0])))[0][0]


keys = sorted(groups)
labels = {k: group_label(groups[k]) for k in keys}
E = embed([labels[k] for k in keys])
S = E @ E.T

pairs = []
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        if S[i, j] >= SIM_THRESHOLD:
            pairs.append((float(S[i, j]), keys[i], keys[j]))
pairs.sort(reverse=True)

upper = S[np.triu_indices(len(keys), k=1)]
print("similarity quantiles (50/90/99/99.9%):", np.round(np.quantile(upper, [0.5, 0.9, 0.99, 0.999]), 3))
print("candidate pairs:", len(pairs), "-> judged:", min(len(pairs), MAX_PAIRS))
for s, a, b in pairs[:20]:
    print(f"  {s:.3f}  {labels[a]}  ~  {labels[b]}")

similarity quantiles (50/90/99/99.9%): [0.811 0.859 0.925 0.962]
candidate pairs: 134 -> judged: 134
  0.992  payment block  ~  payment blocking
  0.987  Purchase-to-pay  ~  PURCHASE-TO-PAY PROCESS
  0.976  invoice-after-goods-receipt three-way matching  ~  invoice-before-goods-receipt three-way matching
  0.972  purchase order  ~  purchase order item
  0.972  invoice receipt recording  ~  recording of invoice receipt
  0.971  purchasing  ~  purchasing process
  0.970  invoice  ~  invoice receipt
  0.967  invoice handling  ~  Invoice processing
  0.965  invoice creation  ~  Vendor invoice creation
  0.964  service entry sheet  ~  service entry sheet recording
  0.963  invoice  ~  vendor invoice
  0.961  invoice clearance  ~  invoice-clearance event
  0.961  invoice  ~  invoice value
  0.960  Activity completion timestamps  ~  completion timestamp
  0.960  invoice receipt  ~  receipt
  0.959  approvals  ~  approval activities
  0.958  goods receipt  ~  goods receipt recording
  0.958  p

## LLM 판정: 정해진 액션만

쌍마다 새 호출로 한 건씩 판정한다. 근거로 각 표기의 원문 문장을 준다.
- `MERGE`: 같은 대상을 가리키는 표기다 (동의어·약어·표기 변형)
- `DISTINCT`: 관련은 있어도 다른 대상이다 (예: 상위/하위 개념, 활동과 그 산출 문서)
- `UNSURE`: 문서만으로 판단할 수 없다

In [5]:
class MergeDecision(BaseModel):
    action: Literal["MERGE", "DISTINCT", "UNSURE"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str = Field(description="One sentence grounded in the given sentences.")


MERGE_SYSTEM = """You decide whether two surface terms from one business process document denote the same thing.
Answer MERGE only if they are synonyms, abbreviations, or spelling/inflection variants of the same referent.
Answer DISTINCT if they are related but different: broader/narrower concepts, an activity vs. the document or object it produces, a type vs. one of its values, or different attributes.
Answer UNSURE if the sentences do not let you decide.
Do not use outside knowledge to force a merge."""


def evidence(key: str, k: int = 2) -> str:
    sents = []
    for t in groups[key]:
        for m in t["mentions"]:
            if m["sentence"] not in sents:
                sents.append(m["sentence"])
    return "\n".join(f"  - {s}" for s in sents[:k])


def judge_pair(a: str, b: str) -> tuple[MergeDecision | None, str | None]:
    prompt = (
        f"Term A: \"{labels[a]}\"\nSentences with A:\n{evidence(a)}\n\n"
        f"Term B: \"{labels[b]}\"\nSentences with B:\n{evidence(b)}\n\n"
        "Do A and B denote the same thing?"
    )
    return generate_json(prompt, MergeDecision, MERGE_SYSTEM)


decisions = []
for sim, a, b in pairs[:MAX_PAIRS]:
    d, err = judge_pair(a, b)
    decisions.append({"a": a, "b": b, "similarity": round(sim, 4), "decision": d.model_dump() if d else None, "error": err})

print(Counter((x["decision"] or {}).get("action", "ERROR") for x in decisions))

Counter({'DISTINCT': 117, 'MERGE': 17})


## ③ 액션 검증 후 실행

- `MERGE` + confidence ≥ `MERGE_CONF`: 실행 후보
- 병합 전에 PK 충돌 검사: 합쳐질 두 집합 사이에 이미 `DISTINCT`로 판정된 쌍이 있으면 실행하지 않고 사람 큐로
- `UNSURE`, 낮은 confidence, 호출 실패: 사람 큐 (두 term은 따로 남아 다음 단계로 간다)

In [6]:
parent = {k: k for k in keys}


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def members_of(root):
    return {k for k in keys if find(k) == root}


distinct_pairs = {frozenset((x["a"], x["b"])) for x in decisions if (x["decision"] or {}).get("action") == "DISTINCT"}
human_queue = []

for x in decisions:
    a, b, d = x["a"], x["b"], x["decision"]
    if d is None:
        human_queue.append(gate_record(f"{a}|{b}", "p2", "judge_failed", "human", error=x["error"]))
        continue
    if d["action"] == "DISTINCT":
        merge_log.append({"action": "KEEP_DISTINCT", "a": a, "b": b, **d})
        continue
    if d["action"] == "UNSURE" or d["confidence"] < MERGE_CONF:
        human_queue.append(gate_record(f"{a}|{b}", "p2", "ambiguous_merge", "human", labels=[labels[a], labels[b]], judge=d))
        continue
    ra, rb = find(a), find(b)
    if ra == rb:
        continue
    conflict = [tuple(p) for p in distinct_pairs if len(p & members_of(ra)) == 1 and len(p & members_of(rb)) == 1]
    if conflict:
        human_queue.append(gate_record(f"{a}|{b}", "p2", "pk_conflict", "human", conflicts_with=conflict, judge=d))
        continue
    parent[rb] = ra
    merge_log.append({"action": "MERGE", "a": a, "b": b, "labels": [labels[a], labels[b]], **d})

print("merges executed:", sum(e["action"] == "MERGE" for e in merge_log), "| human queue:", Counter(h["reason"] for h in human_queue))
for e in merge_log:
    if e["action"] == "MERGE":
        print(f"  MERGE {e['labels']}  ({e['confidence']:.2f}) {e['reason']}")

merges executed: 16 | human queue: Counter({'pk_conflict': 1})
  MERGE ['Purchase-to-pay', 'PURCHASE-TO-PAY PROCESS']  (0.90) Both terms refer to the same underlying business process that tracks a purchase order from creation to invoice clearance.
  MERGE ['invoice receipt recording', 'recording of invoice receipt']  (1.00) "invoice receipt recording" and "recording of invoice receipt" are syntactic variations referring to the same process step of recording an invoice receipt.
  MERGE ['purchasing', 'purchasing process']  (0.90) In this context, 'purchasing' refers to the business function or activity that defines the 'purchasing process' itself, making them synonymous.
  MERGE ['invoice handling', 'Invoice processing']  (0.90) Both terms refer to the same business activity of managing and processing invoices, used synonymously in the context of invoice workflows.
  MERGE ['Activity completion timestamps', 'completion timestamp']  (0.90) Both terms refer to the recorded time at which a

## normed_term 만들기

병합된 집합마다 하나의 normed_term. PK는 언급이 가장 많은 규칙 key다. 모든 표면 변형, 원래 term_id, 근거 mention, 병합 이력을 붙인다.

In [7]:
clusters = defaultdict(list)
for k in keys:
    clusters[find(k)].append(k)


def mention_count(key):
    return sum(len(m["offsets"]) or 1 for t in groups[key] for m in t["mentions"])


normed_terms = []
for root, ks in clusters.items():
    member_terms = [t for k in ks for t in groups[k]]
    pk = max(ks, key=lambda k: (mention_count(k), -len(k)))
    mentions = [{"term_id": t["term_id"], "span": t["span"], **{f: m[f] for f in ("chunk_id", "offsets", "sentence")}} for t in member_terms for m in t["mentions"]]
    normed_terms.append({
        "normed_id": pk,
        "label": labels[pk],
        "kind": Counter(t["kind"] for t in member_terms).most_common(1)[0][0],
        "surface_variants": sorted({t["span"] for t in member_terms}),
        "rule_keys": sorted(ks),
        "term_ids": sorted(t["term_id"] for t in member_terms),
        "n_mentions": sum(len(m["offsets"]) or 1 for m in mentions),
        "chunk_ids": sorted({m["chunk_id"] for m in mentions}),
        "mentions": mentions,
    })

normed_terms.sort(key=lambda n: n["normed_id"])
assert len({n["normed_id"] for n in normed_terms}) == len(normed_terms), "PK 중복"
print("normed_terms:", len(normed_terms))
for n in sorted(normed_terms, key=lambda n: -len(n["surface_variants"]))[:15]:
    print(f"  {n['normed_id']:<35} {n['surface_variants']}")

normed_terms: 129
  purchase_order_item                 ['order item', 'order items', 'purchase order item', 'purchase order items']
  case                                ['case', 'item case', 'item cases']
  purchasing                          ['Procurement Processes', 'purchasing', 'purchasing process']
  approval                            ['approval decisions', 'approvals']
  completion_timestamp                ['Activity completion timestamps', 'completion timestamp']
  consumption                         ['Actual consumption', 'consumption']
  framework_order                     ['framework order', 'framework orders']
  goods_receipt                       ['Goods receipt', 'goods receipt']
  invoice                             ['invoice', 'invoices']
  invoice_clearance                   ['Invoice clearance', 'invoice clearance']
  invoice_event                       ['invoice events', 'invoice-related events']
  invoice_handling                    ['Invoice processing', 'invoice

## ④ 저장

In [8]:
meta = {
    "doc_id": doc_id,
    "model": GEMINI_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "sim_threshold": SIM_THRESHOLD,
    "max_pairs": MAX_PAIRS,
    "merge_conf": MERGE_CONF,
    "abbreviations": ABBREVIATIONS,
    "n_terms_in": len(terms),
    "n_rule_groups": len(groups),
    "n_candidate_pairs": len(pairs),
    "n_judged": len(decisions),
    "n_llm_merges": sum(e["action"] == "MERGE" for e in merge_log),
    "n_normed_terms": len(normed_terms),
    "n_human_queue": len(human_queue),
}
write_json(OUTPUT_DIR / "normed_terms.json", {"meta": meta, "normed_terms": normed_terms})
write_json(OUTPUT_DIR / "merge_log.json", {"merge_log": merge_log, "decisions": decisions})
write_json(OUTPUT_DIR / "human_queue.json", {"human_queue": human_queue})
meta

{'doc_id': 'bpi2019-purchase-process',
 'model': 'gemini-3.5-flash',
 'embedding_model': 'gemini-embedding-001',
 'sim_threshold': 0.92,
 'max_pairs': 150,
 'merge_conf': 0.8,
 'abbreviations': {},
 'n_terms_in': 162,
 'n_rule_groups': 145,
 'n_candidate_pairs': 134,
 'n_judged': 134,
 'n_llm_merges': 16,
 'n_normed_terms': 129,
 'n_human_queue': 1}